In [1]:
# ================================
# 0. Кітапханаларды импорттау
# ================================
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


In [3]:
# =========================================================
# 1. IMDB деректерін жүктеу және сандық форматқа дайындау
# =========================================================

# Ең жиі кездесетін 20 000 сөзді ғана аламыз (сөздік өлшемі)
max_features = 20000

# Барлық пікір ұзындығын 200-ге теңестіреміз
maxlen = 200

# IMDB датасеті already tokenized (сөздер индекс түрінде)
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

print("Train samples:", len(x_train))
print("Test samples :", len(x_test))

# Пікірлердің ұзындығын бірдей ету: қысқасын 0-мен толтыру, ұзындарын кесу
x_train_200 = pad_sequences(x_train, maxlen=maxlen, padding='post', truncating='post')
x_test_200  = pad_sequences(x_test,  maxlen=maxlen, padding='post', truncating='post')

print("x_train shape:", x_train_200.shape)
print("x_test shape :", x_test_200.shape)


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train samples: 25000
Test samples : 25000
x_train shape: (25000, 200)
x_test shape : (25000, 200)


In [4]:
# ============================================
# 2. Қарапайым RNN моделін құру және оқыту
# ============================================

def build_simple_rnn_model(max_features=20000, maxlen=200, emb_dim=128, rnn_units=64):
    model = Sequential([
        Embedding(input_dim=max_features, output_dim=emb_dim, input_length=maxlen),
        SimpleRNN(rnn_units),
        Dense(1, activation='sigmoid')  # бинарлық классификация: оң/теріс
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

simple_rnn_model = build_simple_rnn_model(max_features=max_features, maxlen=maxlen)
simple_rnn_model.summary()

history_rnn = simple_rnn_model.fit(
    x_train_200, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

rnn_test_loss, rnn_test_acc = simple_rnn_model.evaluate(x_test_200, y_test, verbose=0)
print(f"\nSimpleRNN Test Accuracy: {rnn_test_acc:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 98ms/step - accuracy: 0.5025 - loss: 0.6957 - val_accuracy: 0.5064 - val_loss: 0.6945
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 32s 101ms/step - accuracy: 0.5763 - loss: 0.6749 - val_accuracy: 0.5070 - val_loss: 0.6997
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 40s 100ms/step - accuracy: 0.6222 - loss: 0.6169 - val_accuracy: 0.5008 - val_loss: 0.7265
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 31s 100ms/step - accuracy: 0.6737 - loss: 0.5137 - val_accuracy: 0.5092 - val_loss: 0.8014
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 31s 98ms/step - accuracy: 0.7124 - loss: 0.4411 - val_accuracy: 0.5048 - val_loss: 0.9390

SimpleRNN Test Accuracy: 0.5105


In [5]:
# ============================================
# 3. LSTM және GRU модельдері
# ============================================

def build_lstm_model(max_features=20000, maxlen=200, emb_dim=128, units=64):
    model = Sequential([
        Embedding(input_dim=max_features, output_dim=emb_dim, input_length=maxlen),
        LSTM(units),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_gru_model(max_features=20000, maxlen=200, emb_dim=128, units=64):
    model = Sequential([
        Embedding(input_dim=max_features, output_dim=emb_dim, input_length=maxlen),
        GRU(units),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

lstm_model = build_lstm_model(max_features=max_features, maxlen=maxlen)
gru_model  = build_gru_model(max_features=max_features, maxlen=maxlen)

# Оқыту
history_lstm = lstm_model.fit(
    x_train_200, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

history_gru = gru_model.fit(
    x_train_200, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

# Бағалау
lstm_test_loss, lstm_test_acc = lstm_model.evaluate(x_test_200, y_test, verbose=0)
gru_test_loss, gru_test_acc   = gru_model.evaluate(x_test_200, y_test, verbose=0)

print("\n=== Нәтижелер салыстыру ===")
print(f"SimpleRNN Test Accuracy: {rnn_test_acc:.4f}")
print(f"LSTM      Test Accuracy: {lstm_test_acc:.4f}")
print(f"GRU       Test Accuracy: {gru_test_acc:.4f}")

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 82s 253ms/step - accuracy: 0.5243 - loss: 0.6971 - val_accuracy: 0.4932 - val_loss: 0.6942
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 79s 253ms/step - accuracy: 0.5359 - loss: 0.6868 - val_accuracy: 0.5236 - val_loss: 0.6901
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 79s 254ms/step - accuracy: 0.5879 - loss: 0.6635 - val_accuracy: 0.5378 - val_loss: 0.6888
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 81s 250ms/step - accuracy: 0.6077 - loss: 0.6446 - val_accuracy: 0.5178 - val_loss: 0.6948
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 82s 250ms/step - accuracy: 0.6168 - loss: 0.6186 - val_accuracy: 0.5436 - val_loss: 0.7015
Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 86s 265ms/step - accuracy: 0.5177 - loss: 0.6929 - val_accuracy: 0.5148 - val_loss: 0.6924
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 147s 283ms/step - accuracy: 0.6699 - loss: 0.5794 - val_accuracy: 0.8218 - val_loss: 0.4247
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 84s 267ms/step - accuracy: 0.8866 - loss: 0.2835 -

In [ ]:
# ==========================================================
# 4. Ұзын тізбек (500) кезінде RNN vs LSTM оқу динамикасы
# ==========================================================

maxlen_long = 500

x_train_500 = pad_sequences(x_train, maxlen=maxlen_long, padding='post', truncating='post')
x_test_500  = pad_sequences(x_test,  maxlen=maxlen_long, padding='post', truncating='post')

print("x_train_500 shape:", x_train_500.shape)

# Ұзын тізбекке арналған модельдер
simple_rnn_long = Sequential([
    Embedding(input_dim=max_features, output_dim=128, input_length=maxlen_long),
    SimpleRNN(64),
    Dense(1, activation='sigmoid')
])
simple_rnn_long.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

lstm_long = Sequential([
    Embedding(input_dim=max_features, output_dim=128, input_length=maxlen_long),
    LSTM(64),
    Dense(1, activation='sigmoid')
])
lstm_long.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Ерте тоқтату (қалауыңызша)
es = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history_rnn_long = simple_rnn_long.fit(
    x_train_500, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    callbacks=[es],
    verbose=1
)

history_lstm_long = lstm_long.fit(
    x_train_500, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    callbacks=[es],
    verbose=1
)

# Loss графигі
plt.figure(figsize=(10,5))
plt.plot(history_rnn_long.history['loss'], label='SimpleRNN train loss')
plt.plot(history_rnn_long.history['val_loss'], label='SimpleRNN val loss')
plt.plot(history_lstm_long.history['loss'], label='LSTM train loss')
plt.plot(history_lstm_long.history['val_loss'], label='LSTM val loss')
plt.title('Ұзын тізбек (maxlen=500): SimpleRNN vs LSTM')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# Қысқа қорытынды шығару
rnn_long_acc = simple_rnn_long.evaluate(x_test_500, y_test, verbose=0)[1]
lstm_long_acc = lstm_long.evaluate(x_test_500, y_test, verbose=0)[1]

print(f"SimpleRNN (len=500) Test Accuracy: {rnn_long_acc:.4f}")
print(f"LSTM      (len=500) Test Accuracy: {lstm_long_acc:.4f}")

if lstm_long_acc > rnn_long_acc:
    print("\nҚорытынды: LSTM ұзын тізбектерді жақсырақ меңгереді (vanishing gradient әсері аз).")
else:
    print("\nҚорытынды: Бұл жүгірісте айырмашылық аз/кері болуы мүмкін, бірақ теория бойынша LSTM ұзын тәуелділіктерде тұрақтырақ.")

x_train_500 shape: (25000, 500)
Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 67s 207ms/step - accuracy: 0.5076 - loss: 0.6951 - val_accuracy: 0.5118 - val_loss: 0.6943
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 84s 213ms/step - accuracy: 0.5172 - loss: 0.6944 - val_accuracy: 0.5032 - val_loss: 0.6954
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 65s 208ms/step - accuracy: 0.5157 - loss: 0.6914 - val_accuracy: 0.4936 - val_loss: 0.6952
Epoch 1/5
 17/313 ━━━━━━━━━━━━━━━━━━━━ 2:54 591ms/step - accuracy: 0.5067 - loss: 0.6939

In [7]:
# ============================================
# 5. Bidirectional LSTM моделі
# ============================================

bilstm_model = Sequential([
    Embedding(input_dim=max_features, output_dim=128, input_length=200),
    Bidirectional(LSTM(64)),
    Dense(1, activation='sigmoid')
])

bilstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
bilstm_model.summary()

history_bilstm = bilstm_model.fit(
    x_train_200, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

bilstm_test_loss, bilstm_test_acc = bilstm_model.evaluate(x_test_200, y_test, verbose=0)

print("\n=== Соңғы салыстыру (maxlen=200) ===")
print(f"SimpleRNN  Accuracy: {rnn_test_acc:.4f}")
print(f"LSTM       Accuracy: {lstm_test_acc:.4f}")
print(f"GRU        Accuracy: {gru_test_acc:.4f}")
print(f"BiLSTM     Accuracy: {bilstm_test_acc:.4f}")

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 153s 473ms/step - accuracy: 0.7642 - loss: 0.4843 - val_accuracy: 0.8416 - val_loss: 0.3803
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 147s 470ms/step - accuracy: 0.8967 - loss: 0.2662 - val_accuracy: 0.8578 - val_loss: 0.3421
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 149s 475ms/step - accuracy: 0.9354 - loss: 0.1735 - val_accuracy: 0.8346 - val_loss: 0.4780
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 147s 469ms/step - accuracy: 0.9571 - loss: 0.1223 - val_accuracy: 0.8662 - val_loss: 0.4285
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 146s 468ms/step - accuracy: 0.9474 - loss: 0.1567 - val_accuracy: 0.8538 - val_loss: 0.4443

=== Соңғы салыстыру (maxlen=200) ===
SimpleRNN  Accuracy: 0.5105
LSTM       Accuracy: 0.5385
GRU        Accuracy: 0.8392
BiLSTM     Accuracy: 0.8228


Қорытынды сұрақтарға қысқа жауап
1) Vanishing gradient неге пайда болады?
RNN-де backpropagation through time кезінде градиент әр уақыт қадамында туындыларға көбейтіледі. Егер бұл мәндер 1-ден кіші болса, көп қадамнан кейін градиент нөлге жуықтайды. Соның кесірінен тізбектің басындағы ақпарат соңына жетпей қалады, модель ұзақ тәуелділікті үйрене алмайды.

2) LSTM қақпалары қалай көмектеседі?
Forget gate – ескі күйден ненің сақталатынын/ұмытылатынын шешеді.

Input gate – жаңа ақпараттың қандай бөлігі ұяшық күйіне жазылатынын анықтайды.

Output gate – ағымдағы жасырын күйге ненің шығарылатынын басқарады.

Осы механизмдер ақпарат ағынын реттеп, градиенттің толық жоғалуын азайтады, сондытан ұзақ мерзімді тәуелділіктер жақсы сақталады.

3) GRU мен LSTM айырмашылығы және неге GRU таңдалады?
GRU-де қақпа аз (update, reset), жеке cell state жоқ (әдетте ықшам).

Параметр саны аздау → жылдамырақ оқиды, жадыны аз пайдаланады.

Ресурс шектеулі жағдайда (GPU/уақыт аз) GRU көбіне тиімді компромисс: сапасы LSTM-ге жақын, бірақ есептеуі жеңіл.